In [1]:
import os 
%pwd

'd:\\eval-forge2\\notebooks'

In [2]:
os.chdir("../")

In [3]:
import os
import asyncio
import time
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI
from deepeval.models import DeepEvalBaseLLM
from dotenv import load_dotenv

load_dotenv()

class GeminiJudge(DeepEvalBaseLLM):
    def __init__(self, model: str = "gemini-3.1-flash"):
        self.model_name_str = model
        self._model = ChatGoogleGenerativeAI(
            model=model,
            temperature=0,
            api_key=os.getenv("GEMINI_API_KEY"),
        )

    def load_model(self):
        return self._model

    def generate(self, prompt: str, schema: BaseModel = None) -> str | BaseModel:
        time.sleep(0.5)
        chat_model = self.load_model()

        if schema:
            structured = chat_model.with_structured_output(schema)
            return structured.invoke(prompt)

        return chat_model.invoke(prompt).content

    async def a_generate(self, prompt: str, schema: BaseModel = None) -> str | BaseModel:
        await asyncio.sleep(0.5)
        chat_model = self.load_model()

        if schema:
            structured = chat_model.with_structured_output(schema)
            return await structured.ainvoke(prompt)

        res = await chat_model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return f"Gemini ({self.model_name_str})"


judge = GeminiJudge(model="gemini-3.1-flash-lite")
print(judge.generate("Reply with only the word: working"))

[{'type': 'text', 'text': 'working', 'extras': {'signature': 'EjQKMgEMOdbHahZNwkqOj6xt5jCPVzKmK97g3RCkz/i+SIZ68xU+d4+cwDzf8q51wPAU+Kxz'}}]


In [7]:
import pandas as pd

phi_df = pd.read_csv('metrics/phi_metrics.csv')
groq_df     = pd.read_csv('metrics/groq_metrics.csv')
mistral_df   = pd.read_csv('metrics/mistral_metrics.csv')

# Load ground truth
benchmark_df = pd.read_json('data/benchmark.json')

print("miscrosoft_phi:", len(phi_df), "rows")
print("Groq/qwen    :", len(groq_df),     "rows")
print("Mistral-small  :", len(mistral_df),   "rows")
print("GT rows :", len(benchmark_df))
print("\nBenchmark columns:", benchmark_df.columns.tolist())

miscrosoft_phi: 30 rows
Groq/qwen    : 30 rows
Mistral-small  : 30 rows
GT rows : 30

Benchmark columns: ['id', 'category', 'difficulty', 'prompt', 'expected_answer']


In [8]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
import time

# These questions have no fixed ground truth — constraint following only
CONSTRAINT_IDS = {21, 23, 26, 27, 30}

def run_deepeval(df, model_name, benchmark_df, judge):

    print(f"benchmark rows: {len(benchmark_df)} | df rows: {len(df)}")
    assert len(df) == len(benchmark_df), "Row count mismatch!"

    answer_relevancy_metric = AnswerRelevancyMetric(
        threshold=0.5,
        model=judge
    )
    faithfulness_metric = FaithfulnessMetric(
        threshold=0.5,
        model=judge
    )

    ar_scores, f_scores, q_types = [], [], []

    for i in range(len(df)):
        question     = df["question"].iloc[i]
        answer       = df["answer"].iloc[i]
        ground_truth = benchmark_df["expected_answer"].iloc[i]
        question_id  = int(benchmark_df["id"].iloc[i])
        is_constraint = question_id in CONSTRAINT_IDS

        test_case = LLMTestCase(
            input=question,
            actual_output=str(answer),
            expected_output=str(ground_truth),
            retrieval_context=[str(ground_truth)]
        )

        # ── Answer Relevancy — run for ALL questions ─────────────────
        try:
            answer_relevancy_metric.measure(test_case)
            score = answer_relevancy_metric.score
            ar_scores.append(score if score is not None else 0.0)
        except Exception as e:
            print(f"[{model_name}] AR error row {i} (id={question_id}): {e}")
            ar_scores.append(0.0)

        # ── Faithfulness — only for factual questions ─────────────────
        if is_constraint:
            f_scores.append(None)   # NULL in SQLite — not applicable
            q_types.append("constraint")
        else:
            try:
                faithfulness_metric.measure(test_case)
                score = faithfulness_metric.score
                f_scores.append(score if score is not None else 0.0)
            except Exception as e:
                print(f"[{model_name}] Faith error row {i} (id={question_id}): {e}")
                f_scores.append(0.0)
            q_types.append("factual")

        ar_display = f"{ar_scores[-1]:.2f}" if ar_scores[-1] is not None else "N/A"
        f_display  = f"{f_scores[-1]:.2f}"  if f_scores[-1]  is not None else "N/A"
        print(f"[{model_name}] [{i+1}/{len(df)}] (id={question_id}) "
              f"type={'CONSTRAINT' if is_constraint else 'factual  '} | "
              f"AR: {ar_display} | Faith: {f_display}")

        time.sleep(2)

    df = df.copy()
    df["answer_relevancy"] = ar_scores
    df["faithfulness"]     = f_scores
    df["question_type"]    = q_types
    return df

In [ ]:
# Run one at a time so if one fails you don't lose the others
print("=== Running Ms-Phi ===")
deepseek_eval = run_deepeval(phi_df, "ms-phi", benchmark_df, judge)
deepseek_eval.to_csv("evaluations/phi_eval.csv", index=False)
print("Saved phi_eval.csv\n")


Output()

Output()

[ms-phi] [1/30] (id=1) type=factual   | AR: 1.00 | Faith: 0.90


Output()

Output()

...

Output()

Output()

[ms-phi] [29/30] (id=29) type=factual   | AR: 1.00 | Faith: 1.00


Output()

[ms-phi] [30/30] (id=30) type=CONSTRAINT | AR: 1.00 | Faith: N/A
Saved deepseek_eval.csv



In [14]:
print("=== Running Groq Qwen ===")
groq_eval = run_deepeval(groq_df, "groq/qwen3-32b", benchmark_df, judge)
groq_eval.to_csv("evaluations/groq_eval.csv", index=False)
print("Saved groq_eval.csv\n")

Output()

=== Running Groq Qwen ===
benchmark rows: 30 | df rows: 30


Output()

[groq/qwen3-32b] [1/30] (id=1) type=factual   | AR: 1.00 | Faith: 1.00


Output()

...

Output()

Output()

[groq/qwen3-32b] [29/30] (id=29) type=factual   | AR: 1.00 | Faith: 1.00


Output()

[groq/qwen3-32b] [30/30] (id=30) type=CONSTRAINT | AR: 1.00 | Faith: N/A
Saved groq_eval.csv



In [ ]:
print("=== Running Mistral ===")
gemini_eval = run_deepeval(mistral_df, "mistral-small-2506", benchmark_df, judge)
gemini_eval.to_csv("evaluations/mistral_eval.csv", index=False)
print("Saved mistral_eval.csv\n")

In [15]:
import sqlite3
import pandas as pd

phi_eval = pd.read_csv('evaluations/phi_eval.csv')
groq_eval     = pd.read_csv('evaluations/groq_eval.csv')
mistral_eval  = pd.read_csv('evaluations/mistral_eval.csv')


combined = pd.concat([phi_eval, groq_eval,
                      mistral_eval], ignore_index=True)

print(f"Total rows : {len(combined)}")
print(f"Models     : {combined['model'].unique().tolist()}")
print(combined.groupby('model')[['latency_sec', 'tokens_per_sec',
                                  'answer_relevancy', 'faithfulness']].mean().round(3))

conn = sqlite3.connect('benchmark.db')
combined.to_sql('benchmark_results', conn, if_exists='replace', index=False)
conn.close()

print(f"\n✅ Pushed {len(combined)} rows → benchmark.db")

Total rows : 90
Models     : ['ms-phi', 'groq/qwen3-32b', 'mistral-small-2506']
                    latency_sec  tokens_per_sec  answer_relevancy  \
model                                                               
groq/qwen3-32b            3.295         423.922             0.920   
mistral-small-2506        2.928         129.726             0.904   
ms-phi                   36.767           6.278             0.919   

                    faithfulness  
model                             
groq/qwen3-32b             0.914  
mistral-small-2506         0.966  
ms-phi                     0.968  

✅ Pushed 90 rows → benchmark.db


In [16]:
# Verify SQLite
conn = sqlite3.connect('benchmark.db')
df = pd.read_sql("SELECT model, COUNT(*) as rows FROM benchmark_results GROUP BY model", conn)
conn.close()
print(df)

                model  rows
0      groq/qwen3-32b    30
1  mistral-small-2506    30
2              ms-phi    30
